In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, shutil, hashlib, subprocess, time
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive')
PARENT_DIR   = DRIVE_ROOT / 'CALSHIFT_Research'
PROJECT_ROOT = PARENT_DIR / 'calshift-research'
CRED_DIR     = DRIVE_ROOT / '.gitcreds'

subprocess.run(['git','config','--global','user.name','Md Anas Biswas'], check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'], check=False)
subprocess.run(['git','config','--global','credential.helper','store'], check=False)

for fn, dest in [('.git-credentials','/root/.git-credentials'),
                 ('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR / fn, CRED_DIR / fn):
        if cand.exists():
            shutil.copy(cand, dest); os.chmod(dest, 0o600); break

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
subprocess.run(['git','pull','--ff-only','--quiet'], check=False)

import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())

Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [2]:
# =============================================================================
# 06_gates_and_model
# Cell 2 - assemble the primary analysis frame
# Amendment 3 C4: alpha 0.05, randomized APS, Mondrian, Arm A, feasible cells.
# =============================================================================
cov = pd.read_parquet(config.PROC_DIR / 'coverage_long_nslkdd.parquet')
lad = pd.read_csv(config.REPORTS_DIR / 'ladder_shift_measures_nslkdd.csv')
FOCAL = json.loads((config.REPORTS_DIR / 'focal_class_record.json').read_text())['focal_class']

d = cov[(cov.alpha == config.ALPHA_PRIMARY) &
        (cov.score == 'aps') &
        (cov.variant == 'mondrian') &
        (cov.feasible) &
        (cov['class'] != '__marginal__')].copy()

d = d.merge(lad[['rung','realization','S_cov','S_sup','S_lab']],
            on=['rung','realization'], how='left')

assert d['S_cov'].notna().all(), 'shift measures failed to merge'
print(f'primary frame: {len(d):,} rows')
print('classes retained:', sorted(d['class'].unique()))
print('S_lab variance:', float(d['S_lab'].var()), '(zero => omitted, Amendment 3 C1)')
print('\ncoverage by protocol and rung')
print(d.groupby(['protocol','rung'])['coverage'].mean().unstack(0).round(4).to_string())

primary frame: 36,000 rows
classes retained: ['DoS', 'Normal', 'Probe', 'R2L']
S_lab variance: 1.5305724073803107e-07 (zero => omitted, Amendment 3 C1)

coverage by protocol and rung
protocol     REC     SHC     TSC
rung                            
0.0       0.9540  0.7272  0.9524
0.2       0.9539  0.6798  0.9526
0.4       0.9539  0.6275  0.9528
0.6       0.9539  0.5835  0.9519
0.8       0.9539  0.4958  0.9516


In [3]:
# =============================================================================
# Cell 3 - GATE G2, model free
# Amendment 2 B4: the effect must be present at rung 0.00, where S_sup = 0.
# A protocol contrast at a fixed rung is a direct estimate and is unaffected by
# the S_cov / S_sup collinearity (Amendment 3 C3.3).
# Paired by (arch, seed, realization). Bootstrap resamples REALIZATIONS, the
# cluster unit.
# =============================================================================
def paired_contrast(frame, cls, rung, a='SHC', b='TSC', B=2000, seed=0):
    w = frame[(frame['class'] == cls) & (frame.rung == rung)]
    p = (w.pivot_table(index=['arch','seed','realization'], columns='protocol',
                       values='coverage')
           .dropna(subset=[a, b]).reset_index())
    p['diff'] = p[a] - p[b]
    rng = np.random.default_rng(seed)
    reals = p['realization'].unique()
    boots = []
    for _ in range(B):
        pick = rng.choice(reals, len(reals), replace=True)
        boots.append(pd.concat([p[p.realization == r] for r in pick])['diff'].mean())
    boots = np.array(boots)
    return {'class': cls, 'rung': rung, f'{a}_mean': float(p[a].mean()),
            f'{b}_mean': float(p[b].mean()), 'diff': float(p['diff'].mean()),
            'ci_lo': float(np.quantile(boots, 0.025)),
            'ci_hi': float(np.quantile(boots, 0.975)), 'n_pairs': int(len(p))}

g2 = paired_contrast(d, FOCAL, 0.00)
print('GATE G2  focal class', FOCAL, 'at rung 0.00 (S_sup = 0)')
print(f"  SHC {g2['SHC_mean']:.4f}   TSC {g2['TSC_mean']:.4f}")
print(f"  difference {g2['diff']:+.4f}  95% CI [{g2['ci_lo']:+.4f}, {g2['ci_hi']:+.4f}]"
      f"  n_pairs {g2['n_pairs']}")
G2_PASS = g2['ci_hi'] < 0
print('  G2:', 'PASS' if G2_PASS else 'FAIL',
      '- effect present with zero support shift' if G2_PASS else
      '- loss attributable to support shift alone')

GATE G2  focal class R2L at rung 0.00 (S_sup = 0)
  SHC 0.1431   TSC 0.9544
  difference -0.8113  95% CI [-0.8157, -0.8069]  n_pairs 600
  G2: PASS - effect present with zero support shift


In [ ]:
# =============================================================================
# Cell 4 - CRITERION 3, model free
# Amendment 2 B3: focal coverage under SHC at least 5 percentage points below
# TSC. Threshold fixed in advance from operational reasoning, not from data.
# =============================================================================
rows = [paired_contrast(d, FOCAL, r) for r in sorted(d.rung.unique())]
c3 = pd.DataFrame(rows)
print(f'CRITERION 3  focal class {FOCAL}, threshold 5 percentage points')
print(c3[['rung','SHC_mean','TSC_mean','diff','ci_lo','ci_hi']].round(4).to_string(index=False))
C3_PASS = bool((c3['diff'] <= -0.05).all() and (c3['ci_hi'] < -0.05).all())
print('\n  criterion 3:', 'PASS' if C3_PASS else 'FAIL')
print(f'  smallest gap {c3["diff"].abs().min():.4f} against a 0.05 bar')

In [ ]:
# =============================================================================
# Cell 5 - collinearity, Amendment 3 C3.2
# =============================================================================
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = pd.DataFrame({'S_cov': d['S_cov'], 'S_sup': d['S_sup']})
X = sm.add_constant(X)
vif = pd.DataFrame({'term': X.columns,
                    'VIF': [variance_inflation_factor(X.values, i)
                            for i in range(X.shape[1])]})
print(vif.round(3).to_string(index=False))
r = float(np.corrcoef(d['S_cov'], d['S_sup'])[0, 1])
print(f'\ncorr(S_cov, S_sup) = {r:.4f}')
JOINT_ONLY = bool(vif[vif.term != 'const']['VIF'].max() > 10)
print('\nAmendment 3 C3.2:', 'VIF > 10, shift interactions reported as JOINTLY '
      'IDENTIFIED ONLY; no individual coefficient interpreted in isolation'
      if JOINT_ONLY else 'VIF acceptable, coefficients individually interpretable')

In [ ]:
# =============================================================================
# Cell 6 - DEVIATION from Amendment 3 C4, recorded
# The specified beta-binomial GLMM requires glmmTMB (R). No pure-Python package
# supports a binomial mixed model on the COUNT interface with crossed random
# effects; statsmodels BinomialBayesMixedGLM needs Bernoulli rows, which would
# expand this design to roughly 21 million rows.
#
# Substitute: binomial GLM with cluster-robust (sandwich) standard errors
# clustered on ladder realization, with architecture and class as fixed effects.
#   - realization is the resampling unit and carries the repeated-measures
#     dependence, which the sandwich handles
#   - overdispersion is absorbed by the robust covariance rather than by a
#     beta-binomial dispersion parameter
#   - architecture (3 levels) and class (4 levels) would give unreliable
#     variance components in any case, and are of direct interest
# The Gaussian secondary analysis retains true random intercepts (cell 8).
# =============================================================================
DEVIATION = ('Amendment 3 C4 specifies a beta-binomial GLMM in glmmTMB. '
             'Substituted a binomial GLM with cluster-robust standard errors '
             'clustered on ladder realization, because no pure-Python package '
             'supports the specified structure on the count interface. '
             'Architecture and class enter as fixed effects. The Gaussian '
             'secondary model retains random intercepts.')
with open(config.REPORTS_DIR / 'deviations.md', 'a') as f:
    f.write(f'\n## notebook 06\n- {DEVIATION}\n')
print(DEVIATION)

In [ ]:
# =============================================================================
# Cell 7 - primary model
# =============================================================================
import statsmodels.formula.api as smf

m = d.copy()
m['proto'] = pd.Categorical(m.protocol, categories=['TSC','REC','SHC'])
m['failures'] = m['n_eval'] - m['n_covered']

fml = ('n_covered + failures ~ proto*S_cov + proto*S_sup + C(arch) + C(cls)')
m = m.rename(columns={'class': 'cls'})
fit = smf.glm(fml, data=m, family=sm.families.Binomial()).fit(
        cov_type='cluster', cov_kwds={'groups': m['realization']})

print(fit.summary().tables[1].as_text())

PRIMARY = 'proto[T.SHC]:S_cov'
b = float(fit.params[PRIMARY]); ci = fit.conf_int().loc[PRIMARY]
p_raw = float(fit.pvalues[PRIMARY])
p_holm = min(1.0, 2 * p_raw)      # Holm across the two protocol contrasts
print(f'\nPRIMARY TERM {PRIMARY}')
print(f'  coef {b:+.4f}  95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]')
print(f'  p {p_raw:.3g}  Holm-adjusted {p_holm:.3g}')
G1_COEF_PASS = bool(ci[1] < 0 and p_holm < 0.05)
print('  criterion 1:', 'PASS' if G1_COEF_PASS else 'FAIL')
# Amendment 3 C3.2: when VIF > 10 the shift interactions are jointly identified
# only. The joint Wald test asks whether SHC coverage depends on shift AT ALL,
# without requiring the two collinear channels to be separated.
JOINT_TERMS = ['proto[T.SHC]:S_cov', 'proto[T.SHC]:S_sup']
wald = fit.wald_test(' = 0, '.join(JOINT_TERMS) + ' = 0', scalar=True)
p_joint = float(wald.pvalue)
print(f'\nJOINT WALD TEST on {JOINT_TERMS}')
print(f'  statistic {float(wald.statistic):.3f}   p {p_joint:.3g}')
JOINT_PASS = bool(p_joint < 0.05)
print('  joint test:', 'PASS' if JOINT_PASS else 'FAIL')
for t in JOINT_TERMS:
    cc = fit.conf_int().loc[t]
    print(f'    {t:22s} {float(fit.params[t]):+.4f}  [{cc[0]:+.4f}, {cc[1]:+.4f}]')

# Amendment 4 D3.2: directional contrast along the observed shift range.
# Estimable even when its components are not, because the weights follow the
# direction in which the data actually vary.
import numpy as _np
dcov = float(d['S_cov'].max() - d['S_cov'].min())
dsup = float(d['S_sup'].max() - d['S_sup'].min())
L = _np.zeros(len(fit.params))
L[list(fit.params.index).index('proto[T.SHC]:S_cov')] = dcov
L[list(fit.params.index).index('proto[T.SHC]:S_sup')] = dsup
est = float(L @ fit.params.values)
se  = float(_np.sqrt(L @ fit.cov_params().values @ L))
lo, hi = est - 1.96*se, est + 1.96*se
print(f'\nAMENDMENT 4 D3.2  directional contrast (logit scale)')
print(f'  S_cov range {dcov:.4f}   S_sup range {dsup:.4f}')
print(f'  Delta_gap {est:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]')
D32_PASS = bool(hi < 0)
print('  D3.2 directional:', 'PASS' if D32_PASS else 'FAIL')

CRIT1_PASS = bool(JOINT_PASS and D32_PASS) if JOINT_ONLY else G1_COEF_PASS
print(f'\nCRITERION 1 ({"Amendment 4 replacement" if JOINT_ONLY else "original wording"}):',
      'PASS' if CRIT1_PASS else 'FAIL')

if JOINT_ONLY:
    print('\n  Amendment 4 applies: VIF > 10, so criterion 1 is the joint test')
    print('  this as a verdict:')
    print('  plus the directional contrast, not the single coefficient.')
    print('  The single coefficient is reported above, not individually identified.')

In [ ]:
# =============================================================================
# Cell 8 - secondary Gaussian mixed model, with random intercepts
# =============================================================================
g = m.copy()
g['delta'] = g['coverage'] - (1 - config.ALPHA_PRIMARY)
g['re'] = g['realization'].astype(str)

try:
    lmm = smf.mixedlm('delta ~ proto*S_cov + proto*S_sup + C(arch) + C(cls)',
                      data=g, groups=g['re']).fit(reml=True)
    print(lmm.summary().tables[1].to_string())
    print('\nGaussian LMM agrees in sign on the primary term:',
          bool(np.sign(lmm.params.get(PRIMARY, np.nan)) == np.sign(b)))
except Exception as e:
    print('Gaussian LMM did not converge:', type(e).__name__, str(e)[:160])

In [ ]:
# =============================================================================
# Cell 9 - verdict
# =============================================================================
verdict = {
    'dataset': 'nslkdd',
    'focal_class': FOCAL,
    'alpha': config.ALPHA_PRIMARY,
    'G2_rung0_contrast': g2,
    'G2_pass': bool(G2_PASS),
    'criterion3_pass': bool(C3_PASS),
    'criterion3_smallest_gap': float(c3['diff'].abs().min()),
    'criterion1_coef': {'term': PRIMARY, 'coef': b,
                        'ci': [float(ci[0]), float(ci[1])],
                        'p_holm': p_holm, 'pass': bool(G1_COEF_PASS),
                        'evaluable': bool(not JOINT_ONLY)},
    'criterion1_joint': {'terms': JOINT_TERMS, 'p': p_joint,
                         'pass': bool(JOINT_PASS)},
    'criterion1_directional': {'delta_gap': est, 'ci': [lo, hi],
                               'pass': bool(D32_PASS),
                               'S_cov_range': dcov, 'S_sup_range': dsup},
    'criterion1_pass': bool(CRIT1_PASS),
    'amendment4_applies': bool(JOINT_ONLY),
    'vif_joint_only': bool(JOINT_ONLY),
    'corr_scov_ssup': r,
    'status': ('within-dataset analysis; the confirmatory pooled model requires '
               'a second environment (Amendment 3 C5)'),
}
(config.REPORTS_DIR / 'gates_verdict_nslkdd.json').write_text(json.dumps(verdict, indent=2))
c3.to_csv(config.REPORTS_DIR / 'criterion3_focal_contrasts.csv', index=False)
print(json.dumps(verdict, indent=2))
print('\nREMINDER: notebook 06 does not confirm H1. Criterion 4 requires '
      'replication in a second environment.')

In [ ]:
# =============================================================================
# Cell 10 - commit
# =============================================================================
def git(*args, show=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if show:
        if r.stdout.strip(): print(r.stdout.strip())
        if r.stderr.strip(): print(r.stderr.strip())
    return r

for s, dd in [('/root/.git-credentials', PARENT_DIR / '.git-credentials'),
              ('/root/.gitconfig',       PARENT_DIR / '.gitconfig')]:
    if os.path.exists(s): shutil.copy(s, dd)

os.chdir(PROJECT_ROOT)
git('add','-A', show=False)
if git('status','--porcelain', show=False).stdout.strip():
    git('commit','-m','nb06: gate tests and within-dataset model')
    rr = git('push','-u','origin','main')
    if rr.returncode: print('PUSH FAILED. Commit is safe locally.')
else:
    print('nothing to commit')
print(git('log','--oneline','-3', show=False).stdout)